# Social RV Research API Example

This notebook demonstrates how to use the Social RV Research API to fetch remote viewing session data and targets.

## Prerequisites

1. A Research API key from Social RV
2. Create a `.env` file in the project root with:
   ```
   RESEARCH_API_KEY=your-api-key-here
   SOCIAL_RV_API_URL=https://social-rv.com
   ```


In [ ]:
# Setup - add src directory to path
import sys
sys.path.insert(0, '../src')

from dotenv import load_dotenv
load_dotenv()

from comparative_judging import SocialRVClient

# Initialize the client
client = SocialRVClient()
print(f"✅ Client initialized, connecting to: {client.base_url}")


## Fetching Sessions

### List Sessions with Pagination


In [ ]:
# Fetch first page of sessions
result = client.list_sessions(
    page=1,
    page_size=10,
    include_low_value=False
)

print(f"📊 Total sessions: {result['total_count']}")
print(f"📄 Page {result['page']} of {result['total_pages']}")
print(f"\n📋 Sessions on this page:")

for session in result['sessions']:
    print(f"  - {session.id[:8]}... | {session.target_coordinate} | CJ Rank: {session.comparative_judging_rank}")


### Get a Single Session by ID


In [ ]:
# Get the first session from our list
if result['sessions']:
    session_id = result['sessions'][0].id
    session = client.get_session(session_id)
    
    print(f"🎯 Session Details:")
    print(f"   ID: {session.id}")
    print(f"   User: {session.user_display_name}")
    print(f"   Coordinate: {session.target_coordinate}")
    print(f"   Submission Time: {session.submission_time}")
    print(f"   CJ Rank: {session.comparative_judging_rank}")
    print(f"   P-Value: {session.p_value}")
    print(f"\n📎 Target Info:")
    print(f"   Target ID: {session.target_id}")
    print(f"   Description: {session.target_description[:100] if session.target_description else 'N/A'}...")
    print(f"   Image URL: {'Yes' if session.target_image_url else 'No'}")
    print(f"\n📁 Session Media: {len(session.session_media_urls)} files")
    print(f"🎭 Decoys: {len(session.decoy_ids)} targets")


### Bulk Fetch Multiple Sessions


In [ ]:
# Get IDs of first 5 sessions
session_ids = [s.id for s in result['sessions'][:5]]

# Bulk fetch
bulk_result = client.get_sessions_by_ids(session_ids)

print(f"📦 Bulk fetch results:")
print(f"   Requested: {len(session_ids)} IDs")
print(f"   Found: {len(bulk_result['found_ids'])} sessions")
print(f"   Missing: {len(bulk_result['missing_ids'])} sessions")


### Fetch All Sessions (with progress)


In [ ]:
# Fetch all sessions (limited to 50 for demo)
def progress(current, total):
    print(f"\r⏳ Fetching: {current}/{total} sessions", end="")

all_sessions = client.fetch_all_sessions(
    include_low_value=False,
    max_sessions=50,  # Limit for demo
    progress_callback=progress
)

print(f"\n✅ Fetched {len(all_sessions)} sessions")


## Fetching Targets

### List Targets


In [ ]:
# Fetch first page of targets
targets_result = client.list_targets(page=1, page_size=10)

print(f"🎯 Total targets: {targets_result['total_count']}")
print(f"📄 Page {targets_result['page']} of {targets_result['total_pages']}")
print(f"\n📋 Targets on this page:")

for target in targets_result['targets']:
    desc_preview = target.description[:50] + '...' if len(target.description) > 50 else target.description
    print(f"  - {target.id[:8]}... | {target.coordinate} | {desc_preview}")


### Fetch Session with All Decoys

This convenience method fetches a session along with its correct target and all decoy targets.


In [ ]:
# Find a session that has decoys (has been through comparative judging)
sessions_with_decoys = [s for s in all_sessions if s.decoy_ids]

if sessions_with_decoys:
    sample_session = sessions_with_decoys[0]
    print(f"📋 Session {sample_session.id[:8]}... has {len(sample_session.decoy_ids)} decoys")
    
    # Fetch the session with all decoys
    full_data = client.get_session_with_decoys(sample_session.id)
    
    print(f"\n🎯 Correct Target:")
    if full_data['target']:
        print(f"   ID: {full_data['target'].id}")
        print(f"   Description: {full_data['target'].description[:80]}...")
    
    print(f"\n🎭 Decoys ({len(full_data['decoys'])})")
    for i, decoy in enumerate(full_data['decoys'][:3]):
        print(f"   {i+1}. {decoy.id[:8]}... - {decoy.description[:50]}...")
    if len(full_data['decoys']) > 3:
        print(f"   ... and {len(full_data['decoys']) - 3} more")
else:
    print("No sessions with decoys found in the sample")


## Next Steps

Now that you can fetch session data, check out:

- **02_export_to_xlsx.ipynb** - Export sessions to Excel for analysis
- **03_run_judging.ipynb** - Run comparative judging on sessions
